# Tetris decision training on one v5e TPU

The notebook kernel may use Python 3.13. The training environment uses `/usr/bin/python3.12` in a separate venv because the library requires Python 3.12. All JAX work runs in child processes, so the notebook kernel never owns the TPU. This first pass keeps its model, data, checkpoints and replay under ephemeral `/content/tetris-smoke`. The optional hours-long run below requires an explicitly configured persistent path and a larger expert dataset.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

CHECKOUT = Path("/content/minifield-training")
SMOKE_ROOT = Path("/content/tetris-smoke")
VENV = SMOKE_ROOT / "venv"
PYTHON = VENV / "bin/python"
SOURCE_REVISION = "1c429806ede9d851d26833a73ddf6341fcb5e098"
BASE_REVISION = "9d2be5519834990d30996f878b6771cccbd24f2c"
assert Path("/usr/bin/python3.12").is_file(), "This host needs Python 3.12"
assert shutil.which("uv"), "This host needs uv"
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)

def run_child(*command: str, cwd: Path | None = None) -> str:
    with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        assert process.stdout is not None
        lines = []
        for line in process.stdout:
            print(line, end="", flush=True)
            lines.append(line)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    return "".join(lines)


In [ ]:
if not (CHECKOUT / ".git").is_dir():
    assert not CHECKOUT.exists(), f"Expected a clean clone path: {CHECKOUT}"
    run_child("git", "clone", "https://github.com/Minifield-Labs/minifield-training.git", str(CHECKOUT))
assert not run_child("git", "status", "--porcelain", cwd=CHECKOUT).strip(), "Checkout has local edits"
run_child("git", "fetch", "origin", SOURCE_REVISION, cwd=CHECKOUT)
run_child("git", "checkout", "--detach", SOURCE_REVISION, cwd=CHECKOUT)
assert run_child("git", "rev-parse", "HEAD", cwd=CHECKOUT).strip() == SOURCE_REVISION
assert (CHECKOUT / "examples/tetris/train.py").is_file()


In [ ]:
if not PYTHON.is_file():
    run_child("uv", "venv", "--python", "/usr/bin/python3.12", str(VENV))
run_child("uv", "pip", "install", "--python", str(PYTHON), "jax[tpu]==0.7.2")
run_child("uv", "pip", "install", "--python", str(PYTHON), ".[numerical,storage,text]", cwd=CHECKOUT)
run_child(str(PYTHON), "-c", "import sys; assert sys.version_info[:2] == (3, 12); print(sys.version)")
probe = "import jax; devices = jax.devices(); assert jax.__version__ == '0.7.2' and len(devices) == 1 and devices[0].platform == 'tpu', devices; print(devices)"
run_child(str(PYTHON), "-c", probe)


In [ ]:
MODEL_DIR = SMOKE_ROOT / "base-model"
download = "from huggingface_hub import snapshot_download; import sys; snapshot_download(repo_id='LiquidAI/LFM2.5-230M-Base', revision=sys.argv[1], allow_patterns=['config.json', 'tokenizer.json', 'model.safetensors'], local_dir=sys.argv[2])"
run_child(str(PYTHON), "-c", download, BASE_REVISION, str(MODEL_DIR), cwd=CHECKOUT)
assert all((MODEL_DIR / name).is_file() for name in ("config.json", "tokenizer.json", "model.safetensors"))


In [ ]:
DATASET = SMOKE_ROOT / "expert-decisions.jsonl"
if not DATASET.exists():
    run_child(str(PYTHON), "-u", "-m", "examples.tetris.prepare",
              "--model-dir", str(MODEL_DIR), "--output", str(DATASET),
              "--games", "8", "--max-ticks", "8", "--seed", "17",
              "--sequence-length", "512", cwd=CHECKOUT)
manifest = json.loads(DATASET.with_suffix(".jsonl.manifest.json").read_text())
assert manifest["samples"] >= 32, manifest
print(f"Prepared {manifest['samples']} expert decisions", flush=True)


The smoke takes 2 updates, saves the complete FP32 state, and plays one learned-policy game for up to 1,000 ticks. `completed_games` reports whether the game actually ended before that cap. Rerunning this cell validates the latest saved checkpoint and skips the warm start when progress already exists.


In [ ]:
CHECKPOINTS = SMOKE_ROOT / "checkpoints"
common = ["--model-dir", str(MODEL_DIR), "--dataset", str(DATASET),
          "--checkpoint-root", str(CHECKPOINTS), "--run-id", "tetris-v5e-smoke-1",
          "--platform", "tpu", "--checkpoint-every", "2", "--report-every", "1"]
run_child(str(PYTHON), "-u", "-m", "examples.tetris.train", *common,
          "--resume-latest", "--skip-if-resumed", "--max-steps", "2",
          "--eval-games", "1", "--eval-max-ticks", "1000", cwd=CHECKOUT)
assert (CHECKPOINTS / "step-00000002" / "manifest.json").is_file()
print(CHECKPOINTS / "replays" / "step-00000002-game-0.txt", flush=True)


In [ ]:
run_child(str(PYTHON), "-u", "-m", "examples.tetris.train", *common,
          "--resume-latest", "--max-steps", "2", "--eval-games", "0", cwd=CHECKOUT)
assert (CHECKPOINTS / "step-00000004" / "manifest.json").is_file()


The smoke checkpoints and replay live under `/content/tetris-smoke` and disappear with this Colab runtime. Each full checkpoint is about 2.75 GB. Read `checkpoints/replays/step-00000002-game-0.txt` for the board and action trace.

The optional hours-long run starts a separate experiment with a substantial expert dataset and its own checkpoint identity. Set `PERSISTENT_ROOT` to an existing mounted persistent directory first. It never resumes the 64-record smoke state.


In [ ]:
PERSISTENT_ROOT: Path | None = None  # Set to an existing mounted absolute path.
if PERSISTENT_ROOT is None:
    print("Set PERSISTENT_ROOT before starting the optional long run.", flush=True)
else:
    assert PERSISTENT_ROOT.is_absolute() and PERSISTENT_ROOT.is_dir()
    long_model = PERSISTENT_ROOT / "base-model"
    run_child(str(PYTHON), "-c", download, BASE_REVISION, str(long_model), cwd=CHECKOUT)
    long_data = PERSISTENT_ROOT / "expert-decisions.jsonl"
    if not long_data.exists():
        run_child(str(PYTHON), "-u", "-m", "examples.tetris.prepare",
                  "--model-dir", str(long_model), "--output", str(long_data),
                  "--games", "80", "--max-ticks", "5000", "--seed", "17",
                  "--sequence-length", "512", cwd=CHECKOUT)
    run_child(str(PYTHON), "-u", "-m", "examples.tetris.train",
              "--model-dir", str(long_model), "--dataset", str(long_data),
              "--checkpoint-root", str(PERSISTENT_ROOT / "checkpoints"),
              "--run-id", "tetris-base-expert-long-v1", "--platform", "tpu",
              "--resume-latest", "--max-hours", "3", "--checkpoint-every", "200",
              "--report-every", "10", "--eval-games", "2",
              "--eval-max-ticks", "2000", cwd=CHECKOUT)
